# NaniGPT - LoRA fine-tune Gemma 4 E4B for pill organizer detection

This notebook reproduces the per-slot pill organizer classification result reported in the writeup. Gemma 4 E4B is fine-tuned with Unsloth QLoRA on 200 synthetic pill organizer photos, then evaluated on a held-out set of 30 images.

**Runtime:** free Colab T4 (16 GB VRAM) is sufficient. Training takes about 20 minutes at the default `max_steps=120`.

**Data:** 200 training and 30 evaluation pill organizer images with structured ground-truth labels. Generated synthetically with varied lighting, fill patterns, blur, rotation, and titles. Bundled in `training_set.zip` in the repository root.


## Step 1 - Install dependencies

In [ ]:
%%capture
try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
except: _numpy = "numpy"; _pil = "pillow"
!uv pip install -qqq \
 "torch>=2.8.0" "triton>=3.4.0" {_numpy} {_pil} torchvision bitsandbytes \
 unsloth "unsloth_zoo>=2026.4.6" transformers==5.5.0 torchcodec timm \
 trl peft accelerate datasets

## Step 2 - Upload training_set.zip and unzip

In [ ]:
# !unzip '/content/training_set.zip'

In [ ]:
from google.colab import files
print('Upload training_set.zip from your workspace folder:')
uploaded = files.upload()
zip_name = list(uploaded.keys())[0]
!unzip -qo {zip_name} -d /content/data/
# The zip preserves absolute paths - flatten
import os, shutil
for sub in ['training', 'eval']:
 src = '/content/data'
 found = []
 for root, _, files_ in os.walk(src):
 if root.endswith('/' + sub):
 found.append(root)
 if found and not os.path.exists(f'/content/{sub}'):
 shutil.move(found[0], f'/content/{sub}')
!ls /content/training | head -3
!ls /content/eval | head -3
print('Train images:', len([f for f in os.listdir('/content/training') if f.endswith('.jpg')]))
print('Eval images: ', len([f for f in os.listdir('/content/eval') if f.endswith('.jpg')]))

Upload training_set.zip from your workspace folder:


Saving training_set.zip to training_set.zip
index.jsonl
pill_0000.jpg
pill_0001.jpg
index.jsonl
pill_0000.jpg
pill_0001.jpg
Train images: 200
Eval images: 30


## Step 3 - Load Gemma 4 E4B with Unsloth + apply vision LoRA adapters

In [ ]:
import torch
from unsloth import FastModel

model, tokenizer = FastModel.from_pretrained(
 model_name = 'unsloth/gemma-4-e4b-it',
 max_seq_length = 4096,
 load_in_4bit = True,
 full_finetuning = False,
)

model = FastModel.get_peft_model(
 model,
 finetune_vision_layers = True,
 finetune_language_layers = True,
 finetune_attention_modules = True,
 finetune_mlp_modules = True,
 r = 16,
 lora_alpha = 16,
 lora_dropout = 0,
 bias = 'none',
 random_state = 3407,
)
model.print_trainable_parameters()

 Unsloth: Will patch your computer to enable 2x faster free finetuning.
 Unsloth Zoo will now patch everything to make training faster!
==((====))== Unsloth 2026.5.2: Fast Gemma4 patching. Transformers: 5.5.0.
 \\ /| Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \ Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\ / Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-" Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files: 0%| | 0/3 [00:00<?, ?it/s]

Loading weights: 0%| | 0/2130 [00:00<?, ?it/s]

generation_config.json: 0%| | 0.00/203 [00:00<?, ?B/s]

processor_config.json: 0.00B [00:00, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0%| | 0.00/32.2M [00:00<?, ?B/s]

trainable params: 41,222,144 || all params: 8,037,378,592 || trainable%: 0.5129


## Step 4 - Build the training dataset (image + prompt → JSON answer)

In [ ]:
import json, os
from PIL import Image
from datasets import Dataset

TRAIN_INDEX = '/content/training/index.jsonl'
TRAIN_DIR = '/content/training'

rows = []
with open(TRAIN_INDEX) as f:
 for line in f:
 rows.append(json.loads(line))
print('Loaded', len(rows), 'training examples')

def to_messages(row):
 img = Image.open(os.path.join(TRAIN_DIR, row['image'])).convert('RGB')
 img.thumbnail((768, 768))
 return {
 'messages': [
 {'role': 'user', 'content': [
 {'type': 'image', 'image': img},
 {'type': 'text', 'text': row['prompt']},
 ]},
 {'role': 'assistant', 'content': [
 {'type': 'text', 'text': row['answer']},
 ]},
 ]
 }

train_ds = Dataset.from_list([to_messages(r) for r in rows])
print(train_ds)

Loaded 200 training examples
Dataset({
 features: ['messages'],
 num_rows: 200
})


## Step 5 - Configure SFTTrainer and train

T4 with 4-bit quantization + LoRA: per-step ~6-8 sec for image+text. `max_steps=120` is ~15-20 min. Bump to 200-300 for higher quality if you have time.

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.trainer import UnslothVisionDataCollator

trainer = SFTTrainer(
 model = model,
 tokenizer = tokenizer,
 data_collator = UnslothVisionDataCollator(model, tokenizer),
 train_dataset = train_ds,
 args = SFTConfig(
 per_device_train_batch_size = 1,
 gradient_accumulation_steps = 4,
 warmup_steps = 10,
 max_steps = 120,
 learning_rate = 2e-4,
 logging_steps = 5,
 optim = 'adamw_8bit',
 weight_decay = 0.01,
 lr_scheduler_type = 'linear',
 seed = 3407,
 output_dir = '/content/lora_out',
 remove_unused_columns = False,
 dataset_text_field = '',
 dataset_kwargs = {'skip_prepare_dataset': True},
 max_length = 2048,
 report_to = 'none',
 ),
)
trainer_stats = trainer.train()

Unsloth: Model does not have a default image size - using 512


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 2}.
==((====))== Unsloth - 2x faster free finetuning | Num GPUs used = 1
 \\ /| Num examples = 200 | Num Epochs = 3 | Total steps = 120
O^O/ \_/ \ Batch size per device = 1 | Gradient accumulation steps = 4
\ / Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-" Trainable parameters = 41,222,144 of 8,037,378,592 (0.51% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Caching is incompatible with gradient checkpointing in Gemma4TextDecoderLayer. Setting `past_key_values=None`.


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
5,4.071153
10,2.518650
15,0.765778
20,0.240054
25,0.062897
30,0.020939
35,0.014213
40,0.013270
45,0.010923
50,0.008296


Unsloth: Restored added_tokens_decoder metadata in /content/lora_out/checkpoint-120/tokenizer_config.json.


## Step 6 - Save the LoRA adapter (this is what we ship with the submission)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
ADAPTER_DIR = '/content/drive/MyDrive/nanigpt_pill_lora'
trainer.save_model(ADAPTER_DIR)

import os
print('Files saved:')
for f in sorted(os.listdir(ADAPTER_DIR)):
 print(f' {f}: {os.path.getsize(os.path.join(ADAPTER_DIR,f))/1e6:.2f} MB')


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/nanigpt_pill_lora/tokenizer_config.json.


Files saved:
 README.md: 0.01 MB
 adapter_config.json: 0.00 MB
 adapter_model.safetensors: 165.01 MB
 chat_template.jinja: 0.02 MB
 processor_config.json: 0.00 MB
 tokenizer.json: 32.17 MB
 tokenizer_config.json: 0.01 MB
 training_args.bin: 0.01 MB


In [ ]:
!zip -qr nanigpt_pill_lora.zip /content/drive/MyDrive/nanigpt_pill_lora

In [ ]:
from google.colab import files
files.download('nanigpt_pill_lora.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
%cd /content
!git clone --depth 1 https://github.com/ggerganov/llama.cpp
%cd /content/llama.cpp
!pip install -q -r requirements.txt

# Build only the binaries we need
!cmake -B build -DGGML_CUDA=ON -DLLAMA_CURL=OFF 2>&1 | tail -3
!cmake --build build -j 4 --config Release --target llama-quantize llama-export-lora 2>&1 | tail -3
!ls -lh build/bin/llama-quantize build/bin/llama-export-lora

/content
Cloning into 'llama.cpp'...
remote: Enumerating objects: 3039, done.
remote: Counting objects: 100% (3039/3039), done.
remote: Compressing objects: 100% (2382/2382), done.
remote: Total 3039 (delta 628), reused 2298 (delta 581), pack-reused 0 (from 0)
Receiving objects: 100% (3039/3039), 32.41 MiB | 30.67 MiB/s, done.
Resolving deltas: 100% (628/628), done.
/content/llama.cpp
 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.5 MB/s eta 0:00:00
 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 24.7 MB/s eta 0:00:00
 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 173.7 MB/s eta 0:00:00
 Preparing metadata (setup.py) ... done
 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 118.8 MB/s eta 0:00:00
 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 235.9 MB/s eta 0:00:00
 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.5/118.5 kB 19.7 MB/s eta 0:00:00
 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.2/295.2 kB 44.1 MB/s eta 0:00:00
 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
!pip install -q huggingface_hub
from huggingface_hub import hf_hub_download

# Try Unsloth's pre-converted; if 404, try ggml-org/gemma-4-e4b-it-gguf, or google/gemma-4-e4b-it-gguf
base_gguf = hf_hub_download(
 repo_id='unsloth/gemma-4-E4B-it-GGUF',
 filename='gemma-4-E4B-it-BF16.gguf', # or F16/F32 - any non-quantized
 local_dir='/content',
)
print('Downloaded:', base_gguf)
!ls -lh /content/*.gguf

gemma-4-E4B-it-BF16.gguf: 0%| | 0.00/15.1G [00:00<?, ?B/s]

Downloaded: /content/gemma-4-E4B-it-BF16.gguf
-rw-r--r-- 1 root root 15G May 7 19:27 /content/gemma-4-E4B-it-BF16.gguf


In [ ]:
from huggingface_hub import snapshot_download
import os

base_dir = snapshot_download(
 repo_id='unsloth/gemma-4-e4b-it',
 local_dir='/content/base_hf',
 allow_patterns=['*.json', 'tokenizer*', '*.txt'], # config + tokenizer only, NO weights
)
print('Files in base_hf:')
for f in sorted(os.listdir(base_dir)):
 print(f' {f}: {os.path.getsize(os.path.join(base_dir, f))/1024:.1f} KB')

Fetching 5 files: 0%| | 0/5 [00:00<?, ?it/s]

Files in base_hf:
 .cache: 4.0 KB
 config.json: 5.1 KB
 generation_config.json: 0.2 KB
 processor_config.json: 1.6 KB
 tokenizer.json: 31415.7 KB
 tokenizer_config.json: 19.4 KB


In [ ]:
import json, os, shutil
from safetensors.torch import load_file, save_file

ADAPTER_IN = '/content/drive/MyDrive/nanigpt_pill_lora'
BASE_IN = '/content/base_hf'

# ===== 1. Inspect what we have (full names this time) =====
print('=== Base config (multimodal) ===')
with open(os.path.join(BASE_IN, 'config.json')) as f:
 base_cfg = json.load(f)
print(' architectures :', base_cfg.get('architectures'))
print(' model_type :', base_cfg.get('model_type'))
print(' text_config.architectures (if any):', base_cfg.get('text_config', {}).get('architectures'))

print('\n=== LoRA tensor names (sample, FULL) ===')
w = load_file(os.path.join(ADAPTER_IN, 'adapter_model.safetensors'))
print(f' total tensors: {len(w)}')
sample_keys = list(w.keys())[:8]
for k in sample_keys:
 print(f' {k}')

# Group by which sub-module (language vs vision vs audio)
groups = {'language_model': 0, 'vision_tower': 0, 'audio_tower': 0, 'multi_modal_projector': 0, 'other': 0}
for k in w.keys():
 if 'language_model' in k:
 groups['language_model'] += 1
 elif 'vision_tower' in k:
 groups['vision_tower'] += 1
 elif 'audio_tower' in k:
 groups['audio_tower'] += 1
 elif 'multi_modal_projector' in k:
 groups['multi_modal_projector'] += 1
 else:
 groups['other'] += 1
print('\nTensor counts by sub-module:')
for g, c in groups.items():
 print(f' {g}: {c}')

# ===== 2. Build text-only base config (skip directories this time) =====
BASE_OUT = '/content/base_hf_textonly'
os.makedirs(BASE_OUT, exist_ok=True)
for f in os.listdir(BASE_IN):
 src = os.path.join(BASE_IN, f)
 if f != 'config.json' and os.path.isfile(src):
 shutil.copy(src, BASE_OUT)

# Lift text_config to top level, drop vision/audio
text_cfg = dict(base_cfg.get('text_config', base_cfg))
text_cfg['architectures'] = ['Gemma4ForCausalLM']
text_cfg['model_type'] = base_cfg.get('model_type', 'gemma4')
# Carry over a few fields that may be needed
for k in ('eos_token_id', 'pad_token_id', 'bos_token_id', 'tie_word_embeddings'):
 if k in base_cfg and k not in text_cfg:
 text_cfg[k] = base_cfg[k]

with open(os.path.join(BASE_OUT, 'config.json'), 'w') as f:
 json.dump(text_cfg, f, indent=2)
print(f'\nText-only base config written: arch={text_cfg["architectures"]}, model_type={text_cfg["model_type"]}')

# ===== 3. Filter to language_model tensors only, then rename =====
ADAPTER_OUT = '/content/nanigpt_pill_lora_textonly'
os.makedirs(ADAPTER_OUT, exist_ok=True)

renamed = {}
for k, v in w.items():
 # Keep only language model tensors
 if 'vision_tower' in k or 'audio_tower' in k or 'multi_modal_projector' in k:
 continue
 # Strip the multimodal namespace
 new_k = k.replace('model.language_model.', 'model.').replace('language_model.', '')
 renamed[new_k] = v

print(f'\nFiltered + renamed: {len(renamed)} tensors (from {len(w)})')
print(' sample new names:')
for k in list(renamed.keys())[:5]:
 print(f' {k}')

save_file(renamed, os.path.join(ADAPTER_OUT, 'adapter_model.safetensors'))

# Update adapter_config
with open(os.path.join(ADAPTER_IN, 'adapter_config.json')) as f:
 acfg = json.load(f)
if 'target_modules' in acfg and isinstance(acfg['target_modules'], list):
 new_targets = []
 for tm in acfg['target_modules']:
 if 'vision_tower' in tm or 'audio_tower' in tm or 'multi_modal_projector' in tm:
 continue
 new_targets.append(tm.replace('language_model.', ''))
 acfg['target_modules'] = new_targets
acfg['base_model_name_or_path'] = BASE_OUT
with open(os.path.join(ADAPTER_OUT, 'adapter_config.json'), 'w') as f:
 json.dump(acfg, f, indent=2)

# ===== 4. Retry the conversion =====
print('\n=== Running convert_lora_to_gguf ===\n')
import subprocess
result = subprocess.run([
 'python', '/content/llama.cpp/convert_lora_to_gguf.py',
 ADAPTER_OUT,
 '--base', BASE_OUT,
 '--outfile', '/content/nanigpt-lora.gguf',
], capture_output=True, text=True)
print('STDOUT:', result.stdout[-2000:])
print('STDERR:', result.stderr[-2000:])

if os.path.exists('/content/nanigpt-lora.gguf'):
 print('\nSUCCESS')
 !ls -lh /content/nanigpt-lora.gguf
else:
 print('\nStill failed. Paste the diagnostic output above and we adjust.')

=== Base config (multimodal) ===
 architectures : ['Gemma4ForConditionalGeneration']
 model_type : gemma4
 text_config.architectures (if any): None

=== LoRA tensor names (sample, FULL) ===
 total tensors: 812
 base_model.model.model.language_model.layers.0.mlp.down_proj.lora_A.weight
 base_model.model.model.language_model.layers.0.mlp.down_proj.lora_B.weight
 base_model.model.model.language_model.layers.0.mlp.gate_proj.lora_A.weight
 base_model.model.model.language_model.layers.0.mlp.gate_proj.lora_B.weight
 base_model.model.model.language_model.layers.0.mlp.up_proj.lora_A.weight
 base_model.model.model.language_model.layers.0.mlp.up_proj.lora_B.weight
 base_model.model.model.language_model.layers.0.self_attn.k_proj.lora_A.weight
 base_model.model.model.language_model.layers.0.self_attn.k_proj.lora_B.weight

Tensor counts by sub-module:
 language_model: 588
 vision_tower: 224
 audio_tower: 0
 multi_modal_projector: 0
 other: 0

Text-only base config written: arch=['Gemma4ForCausalLM']

In [ ]:
!grep -nE "class.*Gemma.*Model" /content/llama.cpp/convert_hf_to_gguf.py | head -20

7222:class GemmaModel(TextModel):
7276:class Gemma2Model(TextModel):
7326:class Gemma3Model(TextModel):
7382:class EmbeddingGemma(Gemma3Model):
7456:class Gemma3VisionModel(MmprojModel):
7627:class Gemma3nVisionAudioModel(ConformerAudioModel):
7746:class Gemma3NModel(Gemma3Model):
7890:class Gemma4Model(Gemma3Model):
8017:class Gemma4VisionAudioModel(MmprojModel):


In [ ]:
!sed -n '7885,7900p' /content/llama.cpp/convert_hf_to_gguf.py


 yield from super().modify_tensors(data_torch, name, bid)


@ModelBase.register("Gemma4ForConditionalGeneration")
class Gemma4Model(Gemma3Model):
 model_arch = gguf.MODEL_ARCH.GEMMA4

 def norm_shift(self, name: str) -> float:
 del name # unused
 return 0.0

 def set_vocab(self):
 vocab = gguf.LlamaHfVocab(self.dir_model)
 tokens = []
 scores = []


In [ ]:
import json, shutil, os, subprocess
from safetensors.torch import load_file, save_file

BASE_IN = '/content/base_hf'
BASE_OUT = '/content/base_hf_textonly'

# Wipe and re-copy non-config files (start clean)
shutil.rmtree(BASE_OUT, ignore_errors=True)
os.makedirs(BASE_OUT, exist_ok=True)
for f in os.listdir(BASE_IN):
 src = os.path.join(BASE_IN, f)
 if f != 'config.json' and os.path.isfile(src):
 shutil.copy(src, BASE_OUT)

# Build config: keep multimodal architecture name (only string Gemma4Model registers for),
# but strip the vision/audio configs so the model treats itself as text-only.
with open(os.path.join(BASE_IN, 'config.json')) as f:
 base_cfg = json.load(f)

drop_keys = {
 'vision_config', 'audio_config',
 'image_token_id', 'audio_token_id',
 'boi_token_id', 'eoi_token_id',
 'boa_token_id', 'eoa_token_id', 'eoa_token_index',
}
new_cfg = {k: v for k, v in base_cfg.items() if k not in drop_keys}
new_cfg['architectures'] = ['Gemma4ForConditionalGeneration']

with open(os.path.join(BASE_OUT, 'config.json'), 'w') as f:
 json.dump(new_cfg, f, indent=2)

print('Top-level keys:', sorted(new_cfg.keys()))
print('Architectures :', new_cfg['architectures'])
print('Has text_config:', 'text_config' in new_cfg)
print('Has vision_config:', 'vision_config' in new_cfg)

# Run conversion
result = subprocess.run([
 'python', '/content/llama.cpp/convert_lora_to_gguf.py',
 '/content/nanigpt_pill_lora_textonly',
 '--base', BASE_OUT,
 '--outfile', '/content/nanigpt-lora.gguf',
], capture_output=True, text=True)

print('\n--- STDOUT (last 2000) ---')
print(result.stdout[-2000:])
print('\n--- STDERR (last 2000) ---')
print(result.stderr[-2000:])

if os.path.exists('/content/nanigpt-lora.gguf'):
 print('\nSUCCESS - LoRA-GGUF created')
 !ls -lh /content/nanigpt-lora.gguf
else:
 print('\nStill failed - paste the STDERR and we adjust.')

Top-level keys: ['architectures', 'eos_token_id', 'initializer_range', 'model_type', 'pad_token_id', 'text_config', 'tie_word_embeddings', 'torch_dtype', 'transformers_version', 'unsloth_fixed', 'video_token_id', 'vision_soft_tokens_per_image']
Architectures : ['Gemma4ForConditionalGeneration']
Has text_config: True
Has vision_config: False

--- STDOUT (last 2000) ---


--- STDERR (last 2000) ---
--> F32, shape = {16, 2048}
INFO:hf-to-gguf:blk.8.attn_v.weight.lora_a, torch.float32 --> F32, shape = {2560, 16}
INFO:hf-to-gguf:blk.8.attn_v.weight.lora_b, torch.float32 --> F32, shape = {16, 512}
INFO:hf-to-gguf:blk.9.ffn_down.weight.lora_a, torch.float32 --> F32, shape = {10240, 16}
INFO:hf-to-gguf:blk.9.ffn_down.weight.lora_b, torch.float32 --> F32, shape = {16, 2560}
INFO:hf-to-gguf:blk.9.ffn_gate.weight.lora_a, torch.float32 --> F32, shape = {2560, 16}
INFO:hf-to-gguf:blk.9.ffn_gate.weight.lora_b, torch.float32 --> F32, shape = {16, 10240}
INFO:hf-to-gguf:blk.9.ffn_up.weight.lora_a, tor

In [ ]:
from huggingface_hub import hf_hub_download, list_repo_files

candidates = [
 'unsloth/gemma-4-E4B-it-GGUF',
 'unsloth/gemma-4-e4b-it-GGUF',
 'unsloth/gemma-4-E4B-GGUF',
 'google/gemma-4-e4b-it-gguf',
 'bartowski/gemma-4-E4B-it-GGUF',
 'ggml-org/gemma-4-e4b-it-GGUF',
]

base_gguf = None
for repo in candidates:
 try:
 files = list_repo_files(repo)
 # Prefer F16/BF16 (unquantized) for clean LoRA merge
 unq = [f for f in files if f.endswith('.gguf') and any(x in f.upper() for x in ('F16','BF16','F32'))]
 if not unq:
 unq = [f for f in files if f.endswith('.gguf')]
 if unq:
 print(f'Found in {repo}:')
 for f in unq[:5]:
 print(f' {f}')
 target = unq[0]
 print(f'\nDownloading {target}...')
 base_gguf = hf_hub_download(repo_id=repo, filename=target, local_dir='/content')
 print(f'Done: {base_gguf}')
 break
 except Exception as e:
 print(f' {repo}: {type(e).__name__} - {str(e)[:80]}')

if base_gguf:
 !ls -lh {base_gguf}
else:
 print('\nNo pre-converted base GGUF found. Skip to Step A-fallback.')

Found in unsloth/gemma-4-E4B-it-GGUF:
 gemma-4-E4B-it-BF16.gguf
 mmproj-BF16.gguf
 mmproj-F16.gguf
 mmproj-F32.gguf

Done: /content/gemma-4-E4B-it-BF16.gguf
-rw-r--r-- 1 root root 15G May 7 19:27 /content/gemma-4-E4B-it-BF16.gguf


In [ ]:
import os, subprocess

# Adjust this if Step A used a different filename
print(f'Using base: {base_gguf}')
!ls -lh {base_gguf}

# Merge
print('\nMerging LoRA into base...')
result = subprocess.run([
 '/content/llama.cpp/build/bin/llama-export-lora',
 '--model', base_gguf,
 '--lora', '/content/nanigpt-lora.gguf',
 '--output', '/content/nanigpt-merged-f16.gguf',
], capture_output=True, text=True)
print('STDOUT:', result.stdout[-1500:])
print('STDERR:', result.stderr[-1500:])
!ls -lh /content/nanigpt-merged-f16.gguf

# Quantize to Q4_K_M
print('\nQuantizing to Q4_K_M...')
result = subprocess.run([
 '/content/llama.cpp/build/bin/llama-quantize',
 '/content/nanigpt-merged-f16.gguf',
 '/content/nanigpt-q4_k_m.gguf',
 'Q4_K_M',
], capture_output=True, text=True)
print('STDOUT:', result.stdout[-1500:])
print('STDERR:', result.stderr[-1500:])
!ls -lh /content/nanigpt-q4_k_m.gguf

Using base: /content/gemma-4-E4B-it-BF16.gguf
-rw-r--r-- 1 root root 15G May 7 19:27 /content/gemma-4-E4B-it-BF16.gguf

Merging LoRA into base...
STDOUT: scale=1.000000 rank=16
merge_tensor : + output type is f16
merge_tensor : blk.9.ffn_gate.weight [2560, 10240, 1, 1]
merge_tensor : + dequantize base tensor from bf16 to F32
merge_tensor : + merging from adapter[0] type=f32
merge_tensor : input_scale=1.000000 calculated_scale=1.000000 rank=16
merge_tensor : + output type is f16
copy_tensor : blk.9.ffn_norm.weight [2560, 1, 1, 1]
merge_tensor : blk.9.ffn_up.weight [2560, 10240, 1, 1]
merge_tensor : + dequantize base tensor from bf16 to F32
merge_tensor : + merging from adapter[0] type=f32
merge_tensor : input_scale=1.000000 calculated_scale=1.000000 rank=16
merge_tensor : + output type is f16
copy_tensor : blk.9.inp_gate.weight [2560, 256, 1, 1]
copy_tensor : blk.9.layer_output_scale.weight [1, 1, 1, 1]
copy_tensor : blk.9.post_attention_norm.weight [2560, 1, 1, 1]
copy_tensor : blk.9.p

In [ ]:
import shutil, os
os.makedirs('/content/drive/MyDrive/nanigpt_gguf', exist_ok=True)
shutil.copy('/content/nanigpt-q4_k_m.gguf', '/content/drive/MyDrive/nanigpt_gguf/')
shutil.copy('/content/Modelfile', '/content/drive/MyDrive/nanigpt_gguf/')
shutil.copy('/content/nanigpt-lora.gguf', '/content/drive/MyDrive/nanigpt_gguf/')
print('Copied to Drive.')
!ls -lh /content/drive/MyDrive/nanigpt_gguf/

Copied to Drive.
total 5.2G
-rw------- 1 root root 579 May 7 20:21 Modelfile
-rw------- 1 root root 141M May 7 20:21 nanigpt-lora.gguf
-rw------- 1 root root 5.0G May 7 20:21 nanigpt-q4_k_m.gguf


In [ ]:
from huggingface_hub import logout, login, whoami
# logout() # clears any cached token

import os
# Make sure the env var doesn't override
os.environ.pop('HF_TOKEN', None)
os.environ.pop('HUGGING_FACE_HUB_TOKEN', None)

login(token=input("Paste FRESH WRITE token (not via chat): "), add_to_git_credential=False)

info = whoami()
print('User:', info['name'])
print('Token role:', info['auth']['accessToken'].get('role'))
print('Token name:', info['auth']['accessToken'].get('displayName'))

Paste FRESH WRITE token (not via chat): hf_REDACTED
User: sammy786
Token role: write
Token name: write


In [ ]:
from huggingface_hub import HfApi, login

login(token=input("Paste HF write token: "))

api = HfApi()
repo_id = 'sammy786/nanigpt-gemma4-e4b-pill-lora-gguf'

api.create_repo(repo_id=repo_id, repo_type='model', private=False, exist_ok=True)

for fname in ['nanigpt-q4_k_m.gguf', 'nanigpt-lora.gguf', 'Modelfile']:
 print(f'Uploading {fname}...')
 api.upload_file(
 path_or_fileobj=f'/content/{fname}',
 path_in_repo=fname,
 repo_id=repo_id,
 repo_type='model',
 )
print(f'\nDone. https://huggingface.co/{repo_id}')

Paste HF write token: hf_REDACTED
Uploading nanigpt-q4_k_m.gguf...


Processing Files (0 / 0) : | | 0.00B / 0.00B 

New Data Upload : | | 0.00B / 0.00B 

 /content/nanigpt-q4_k_m.gguf: 1%|1 | 70.7MB / 5.34GB 

Uploading nanigpt-lora.gguf...


Processing Files (0 / 0) : | | 0.00B / 0.00B 

New Data Upload : | | 0.00B / 0.00B 

 /content/nanigpt-lora.gguf : 3%|2 | 3.91MB / 147MB 

Uploading Modelfile...

Done. https://huggingface.co/sammy786/nanigpt-gemma4-e4b-pill-lora-gguf


In [ ]:
import subprocess

# Tiny generation test - just prove the binary loads the model and produces 1 token
proc = subprocess.Popen([
 '/content/llama.cpp/build/bin/llama-cli',
 '-m', '/content/nanigpt-q4_k_m.gguf',
 '-p', 'Hi',
 '-n', '1',
 '-ngl', '99',
], stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)

import time
start = time.time()
while time.time() - start < 90:
 if proc.poll() is not None:
 break
 time.sleep(2)

if proc.poll() is None:
 proc.kill(); proc.wait(timeout=5)
 out, err = proc.communicate()
 print(f'=== Timed out after 90s, killed ===')
else:
 out, err = proc.communicate()
 print(f'=== Completed in {time.time()-start:.1f}s ===')

print('\nSTDOUT:', (out or '')[-1500:])
print('\nSTDERR:', (err or '')[-2500:])

=== Timed out after 90s, killed ===

STDOUT: 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

> 

>

In [ ]:
a=5

## Step 7 - Evaluation: base vs fine-tuned per-slot accuracy

**This is the writeup-defining section.** Run inference on the held-out 30 eval images, score per-slot, compare against the same images run through the **base** model (which we'd need to load separately). For simplicity we score only the fine-tuned model here and use the prior measurement (78%) as the baseline anchor.

In [ ]:
import re, json
from PIL import Image
from transformers import TextStreamer

def classify(img):
 msgs = [
 {'role': 'user', 'content': [
 {'type': 'image', 'image': img},
 {'type': 'text', 'text': 'List EVERY compartment in this weekly pill organizer (MON-SUN, AM and PM = 14 total). For each, output JSON: [{"day":"MON","ampm":"AM","has_pills":true}, ...]. Output ONLY the JSON array, nothing else.'},
 ]},
 ]
 inputs = tokenizer.apply_chat_template(
 msgs, add_generation_prompt=True, tokenize=True,
 return_dict=True, return_tensors='pt',
 ).to(model.device)
 with torch.no_grad():
 out = model.generate(**inputs, max_new_tokens=600, temperature=0.2, top_p=0.9, do_sample=False)
 text = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
 m = re.search(r'\[.*\]', text, re.DOTALL)
 return json.loads(m.group(0)) if m else None

EVAL_DIR = '/content/eval'
EVAL_INDEX = '/content/eval/index.jsonl'

rows = [json.loads(l) for l in open(EVAL_INDEX)]
total = correct = 0
per_image = []

for row in rows:
 img = Image.open(os.path.join(EVAL_DIR, row['image'])).convert('RGB')
 img.thumbnail((768, 768))
 pred = classify(img) or []
 truth = json.loads(row['answer'])
 truth_map = {(s['day'], s['ampm']): s['has_pills'] for s in truth}
 pred_map = {(s['day'], s['ampm']): s['has_pills'] for s in pred}
 img_correct = sum(1 for k in truth_map if pred_map.get(k) == truth_map[k])
 img_total = len(truth_map)
 correct += img_correct
 total += img_total
 per_image.append((row['image'], img_correct, img_total))

print(f'\n=== FINE-TUNED MODEL - held-out eval ===')
print(f'Per-slot accuracy: {correct}/{total} = {100*correct/total:.1f}%')
print(f'Per-image breakdown:')
for f, c, t in per_image[:10]:
 print(f' {f}: {c}/{t}')
print(' ... (showing first 10 of 30)')


=== FINE-TUNED MODEL - held-out eval ===
Per-slot accuracy: 418/420 = 99.5%
Per-image breakdown:
 pill_0000.jpg: 14/14
 pill_0001.jpg: 14/14
 pill_0002.jpg: 14/14
 pill_0003.jpg: 13/14
 pill_0004.jpg: 14/14
 pill_0005.jpg: 14/14
 pill_0006.jpg: 14/14
 pill_0007.jpg: 14/14
 pill_0008.jpg: 14/14
 pill_0009.jpg: 14/14
 ... (showing first 10 of 30)


In [ ]:
from PIL import Image
import json, re

# Test on the original hand-tested image (different generator parameters, treat as held-out OOD)
sanity_files = [
 '/content/eval/pill_0000.jpg', # already in eval set
]

# Better: upload the original test2/test3 from your workspace
print("Upload test2_baseline_full_week.jpg and test3_three_taken_today.jpg:")
from google.colab import files
files.upload()

for fname in ['test2_baseline_full_week.jpg', 'test3_three_taken_today.jpg']:
 img = Image.open(fname).convert('RGB')
 img.thumbnail((768, 768))
 pred = classify(img)
 print(f"\n{fname}:")
 print(json.dumps(pred, indent=2))

Upload test2_baseline_full_week.jpg and test3_three_taken_today.jpg:


Saving test2_baseline_full_week.jpg to test2_baseline_full_week.jpg
Saving test3_three_taken_today.jpg to test3_three_taken_today.jpg

test2_baseline_full_week.jpg:
[
 {
 "day": "MON",
 "ampm": "AM",
 "has_pills": true
 },
 {
 "day": "MON",
 "ampm": "PM",
 "has_pills": true
 },
 {
 "day": "TUE",
 "ampm": "AM",
 "has_pills": true
 },
 {
 "day": "TUE",
 "ampm": "PM",
 "has_pills": true
 },
 {
 "day": "WED",
 "ampm": "AM",
 "has_pills": true
 },
 {
 "day": "WED",
 "ampm": "PM",
 "has_pills": true
 },
 {
 "day": "THU",
 "ampm": "AM",
 "has_pills": true
 },
 {
 "day": "THU",
 "ampm": "PM",
 "has_pills": true
 },
 {
 "day": "FRI",
 "ampm": "AM",
 "has_pills": true
 },
 {
 "day": "FRI",
 "ampm": "PM",
 "has_pills": true
 },
 {
 "day": "SAT",
 "ampm": "AM",
 "has_pills": true
 },
 {
 "day": "SAT",
 "ampm": "PM",
 "has_pills": true
 },
 {
 "day": "SUN",
 "ampm": "AM",
 "has_pills": true
 },
 {
 "day": "SUN",
 "ampm": "PM",
 "has_pills": true
 }
]

test3_three_taken_today.jpg:
[
 {
 "day": "MON"